# 37 - Humanoid Perception Stack

## What / Why / How

**What we are trying to do:** Sketch the perception stack for a humanoid: cameras, depth, segmentation, tracking, pose, and scene memory.

**Why this matters:** A general-purpose robot needs persistent understanding of objects, humans, free space, hands, and task state.

**How we will do it:** Build a toy scene graph from detections, update tracks over time, and query the graph for task-relevant objects.

## Perception Stack

A humanoid needs an object-centric scene representation: what exists, where it is, whether it moved, and how it relates to the task.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
rng = np.random.default_rng(37)
detections_t0 = [
    {"label": "red cup", "xy": np.array([0.5, 0.2]), "confidence": 0.94},
    {"label": "blue bowl", "xy": np.array([0.9, 0.1]), "confidence": 0.91},
    {"label": "person", "xy": np.array([1.5, -0.4]), "confidence": 0.98},
]
tracks = {}
next_id = 0
for det in detections_t0:
    tracks[next_id] = {**det, "age": 1}
    next_id += 1
print(tracks)

In [ ]:
detections_t1 = [
    {"label": "red cup", "xy": np.array([0.54, 0.22]), "confidence": 0.92},
    {"label": "blue bowl", "xy": np.array([0.91, 0.12]), "confidence": 0.88},
    {"label": "person", "xy": np.array([1.45, -0.38]), "confidence": 0.97},
    {"label": "spoon", "xy": np.array([0.75, 0.3]), "confidence": 0.78},
]

def update_tracks(tracks, detections, max_dist=0.2):
    global next_id
    assigned = set()
    for det in detections:
        best = None
        best_dist = float("inf")
        for tid, track in tracks.items():
            if tid in assigned or track["label"] != det["label"]:
                continue
            dist = np.linalg.norm(track["xy"] - det["xy"])
            if dist < best_dist:
                best, best_dist = tid, dist
        if best is not None and best_dist < max_dist:
            tracks[best].update(det)
            tracks[best]["age"] += 1
            assigned.add(best)
        else:
            tracks[next_id] = {**det, "age": 1}
            next_id += 1
    return tracks

tracks = update_tracks(tracks, detections_t1)
for tid, track in tracks.items():
    print(tid, track)

In [ ]:
scene_graph = []
labels = list(tracks.items())
for i, a in labels:
    for j, b in labels:
        if i >= j:
            continue
        dist = np.linalg.norm(a["xy"] - b["xy"])
        if dist < 0.45:
            scene_graph.append((a["label"], "near", b["label"], round(float(dist), 2)))
print(scene_graph)

In [ ]:
instruction = "put the red cup in the blue bowl"
relevant = [track for track in tracks.values() if any(word in track["label"] for word in ["cup", "bowl"])]
print("instruction:", instruction)
print("relevant objects:", relevant)

## Exercises

1. Add object disappearance and memory timeout.
2. Add a table surface and support relations.
3. Explain how bad tracking could cause unsafe manipulation.